## Imports

In [ ]:
import os
import pydicom
import SimpleITK as sitk
import numpy as np
import glob
import shutil
from pathlib import Path
from tqdm import tqdm

## Configuration

**This is the only cell you need to edit.** Set `BASE_PATH` to the folder that contains your DICOM data (`.dcm` or `.IMA` files). Subfolders are searched recursively, so it can point to a single scan or to a folder of many scans. All remaining cells run without further changes.

In [ ]:
# Folder containing your DICOM data (searched recursively).
# Windows example:     BASE_PATH = Path(r"C:\Users\<you>\Documents\my_scans")
# macOS/Linux example: BASE_PATH = Path("/home/<you>/my_scans")
BASE_PATH = Path("path/to/your/dicom_data")

# Converted NIfTI files are collected in this folder (created next to BASE_PATH).
# It is kept outside BASE_PATH so reruns don't re-scan the outputs.
OUTPUT_DIR = BASE_PATH.parent / f"{BASE_PATH.name}_nifti"

if not BASE_PATH.exists():
    raise FileNotFoundError(
        f"BASE_PATH does not exist: {BASE_PATH}\n"
        "Edit BASE_PATH in this cell to point to your DICOM folder before running the rest of the notebook."
    )

## Functions

In [ ]:
# This will convert all dicom files in a directory to a single nifti file and copy the pixel spacing and slice thickness attributes

def convert_dicom_to_nifti(dicom_dir, nifti_file):

    # Accept both .IMA and .dcm (case-insensitive)
    dicom_files = [f for f in os.listdir(dicom_dir) if f.lower().endswith(('.ima', '.dcm'))]
    dicom_files.sort()
    if not dicom_files:
        print(f"No DICOM files (.dcm/.IMA) found in {dicom_dir}; skipping.")
        return

    # Read DICOMs
    dicom_slices = [pydicom.dcmread(os.path.join(dicom_dir, f)) for f in dicom_files]
    dicom_data = [s.pixel_array for s in dicom_slices]

    # Convert to SimpleITK image
    nifti_data = sitk.GetImageFromArray(dicom_data)
    nifti_img = sitk.Cast(nifti_data, sitk.sitkUInt16)

    # Try to copy spacing; fall back to defaults if missing
    # DICOM PixelSpacing is [row spacing, column spacing] = [y, x]
    try:
        y_space, x_space = dicom_slices[0].PixelSpacing
    except Exception:
        y_space = x_space = 1.0
    z_space = getattr(dicom_slices[0], "SliceThickness", 1.0)

    # SimpleITK expects spacing in (x, y, z) order
    nifti_img.SetSpacing([float(x_space), float(y_space), float(z_space)])

    # Write out nifti
    sitk.WriteImage(nifti_img, nifti_file)


# This will return a list of directories containing DICOM files

def get_directories_with_dicom_files(root_dir):
    dicom_dirs = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if any(f.lower().endswith(('.ima', '.dcm')) for f in filenames):
            dicom_dirs.append(dirpath)
    return dicom_dirs

## Generate NIfTI files

In [ ]:
# Given BASE_PATH, this script will find all directories containing DICOM files, convert the DICOM files to a single NIfTI,
# and save each NIfTI file in the same directory as its DICOM files

dicom_dirs = get_directories_with_dicom_files(BASE_PATH)
for dicom_dir in dicom_dirs:
    nifti_file = os.path.join(dicom_dir, os.path.basename(dicom_dir) + ".nii.gz")
    print(nifti_file)
    convert_dicom_to_nifti(dicom_dir, nifti_file)

## Copy NIfTI files to new folder

In [ ]:
# Create the output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find all NIfTI files under BASE_PATH
nifti_files = glob.glob(os.path.join(BASE_PATH, '**', '*.nii.gz'), recursive=True)

# Copy each NIfTI file to the output directory
for nifti_file in nifti_files:
    shutil.copy(nifti_file, OUTPUT_DIR)

print(f"Copied {len(nifti_files)} NIfTI files to {OUTPUT_DIR}")